<a href="https://colab.research.google.com/github/basilhakim/NLP-classworks/blob/main/Q1_Regression_for_median_house_value.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 1 : MCTA 4362 Machine Learning (QUESTION 1)**  
MUHAMMAD BASIL BIN ABDUL HAKIM  
2215315  
2nd April 2026

### Part A: Selection of Independent Variables and Justification

To predict `median_house_value`, the following variables were chosen for their direct impact on real estate pricing:

1.  **Median Income**: The most critical factor. Higher household income directly correlates with higher purchasing power, driving up property values in those areas.
2.  **Location (Latitude & Longitude)**: Real estate is all about 'location'. In California, proximity to major cities (SF, LA) and the coastline significantly inflates housing costs.
3.  **Housing Median Age**: Newer homes often have higher values due to modern standards, while historical areas can maintain high value due to prestige. Age helps the model adjust for these trends.
4.  **Rooms per Household (Derived)**: Instead of raw counts, this density metric tells us how spacious the homes are. Larger, less crowded homes are typically more valuable.
5.  **Ocean Proximity**: Being near the water is a primary premium factor in California. Categorizing areas by their distance to the ocean allows the model to capture this price surge.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv('housing.csv')

# Display first few rows to confirm it loaded correctly
display(df.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


In [ ]:
df = pd.read_csv('housing.csv')

# Check for missing values in each column before preprocessing
missing_values = df.isnull().sum()
print("Missing values per column in the raw dataset:")
print(missing_values[missing_values > 0])

print("\nTotal missing in total_bedrooms:", df['total_bedrooms'].isnull().sum())

Missing values per column in the raw dataset:
total_bedrooms    207
dtype: int64

Total missing in total_bedrooms: 207


In [ ]:
# Data Preprocessing
# Handling Missing Values
# The 'total_bedrooms' column has 207 missing values. We will fill them with the median.
median_bedrooms = df['total_bedrooms'].median()
df['total_bedrooms'] = df['total_bedrooms'].fillna(median_bedrooms)

# Feature Engineering: Creating more logical variables
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

# Encoding Categorical Variables
# 'ocean_proximity' is categorical. We then use One-Hot Encoding (pd.get_dummies)
df_encoded = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

# Define Independent (X) and Dependent (y) variables
X = df_encoded.drop('median_house_value', axis=1)
y = df_encoded['median_house_value']

# Splitting the Data
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
# Algorithms like SVR and Polynomial Regression are very sensitive to unscaled data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preparation complete!")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

Data preparation complete!
Training data shape: (16512, 15)
Testing data shape: (4128, 15)


### Part B: Data Preparation Justification

To ensure the models work accurately, we performed these simple but essential steps:

1. **Handling Missing Values**: We filled missing 'total_bedrooms' with the **median**. This keeps the dataset complete without letting extreme values (outliers) distort the average.
2. **Feature Engineering**: We created ratios like 'rooms per household'. This is more useful than raw counts because it shows how spacious a home actually is, regardless of the suburb size.
3. **Encoding Categories**: We converted 'ocean proximity' into numbers using **One-Hot Encoding**. Computers can't read text, so this turns location names into a format the math models can understand.
4. **Feature Scaling**: we scaled all numbers to a similar range. Since some numbers are in the thousands (like population) and others are small (like income), scaling prevents the large numbers from 'drowning out' the small ones.
5. **Data Splitting**: We kept 20% of the data aside for testing. This acts like a final exam for the model to see if it can predict prices for houses it has never seen before.

In [ ]:
# Model Building and Evaluation (Parts C, D, E)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# --- Part C: Multiple Linear Regression ---
mlr_model = LinearRegression()
mlr_model.fit(X_train_scaled, y_train)
y_pred_mlr = mlr_model.predict(X_test_scaled)

r2_mlr = r2_score(y_test, y_pred_mlr)
mae_mlr = mean_absolute_error(y_test, y_pred_mlr)
rmse_mlr = np.sqrt(mean_squared_error(y_test, y_pred_mlr))

print("--- Part C: Multiple Linear Regression Results ---")
print(f"R²: {r2_mlr:.4f}")
print(f"MAE: ${mae_mlr:,.2f}")
print(f"RMSE: ${rmse_mlr:,.2f}\n")


# --- Part D: Polynomial Regression ---
# To avoid the "curse of dimensionality" and massive overfitting,
# we only apply polynomial features to the most important continuous variables.
core_features = ['median_income', 'housing_median_age', 'latitude', 'longitude']
X_train_core = X_train[core_features]
X_test_core = X_test[core_features]

# Scale core features
scaler_core = StandardScaler()
X_train_core_scaled = scaler_core.fit_transform(X_train_core)
X_test_core_scaled = scaler_core.transform(X_test_core)

# Apply Polynomial Features (Degree = 2)
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train_core_scaled)
X_test_poly = poly.transform(X_test_core_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)

r2_poly = r2_score(y_test, y_pred_poly)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))

print("--- Part D: Polynomial Regression (Degree=2) Results ---")
print(f"R²: {r2_poly:.4f}")
print(f"MAE: ${mae_poly:,.2f}")
print(f"RMSE: ${rmse_poly:,.2f}\n")


# --- Part E: Support Vector Regression (SVR) ---
# We use the RBF (Radial Basis Function) kernel.
# We increase 'C' to 100,000 because target house values are in the hundreds of thousands.
svr_model = SVR(kernel='rbf', C=100000)
svr_model.fit(X_train_scaled, y_train)
y_pred_svr = svr_model.predict(X_test_scaled)

r2_svr = r2_score(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))

print("--- Part E: Support Vector Regression (RBF Kernel) Results ---")
print(f"R²: {r2_svr:.4f}")
print(f"MAE: ${mae_svr:,.2f}")
print(f"RMSE: ${rmse_svr:,.2f}")

--- Part C: Multiple Linear Regression Results ---
R²: 0.5970
MAE: $50,888.66
RMSE: $72,668.54

--- Part D: Polynomial Regression (Degree=2) Results ---
R²: 0.6173
MAE: $50,740.88
RMSE: $70,815.79

--- Part E: Support Vector Regression (RBF Kernel) Results ---
R²: 0.7432
MAE: $38,557.94
RMSE: $58,009.39


### Part F: Final Discussion and Model Comparison

Based on the execution results from Parts C, D, and E, we can compare the performance of the three models:

| Model | R" Score | Mean Absolute Error (MAE) | Root Mean Squared Error (RMSE) |
|---|---|---|---|
| **Multiple Linear Regression** | 0.5970 | $50,888.66 | $72,668.54 |
| **Polynomial Regression (Deg 2)** | 0.6173 | $50,740.88 | $70,815.79 |
| **Support Vector Regression (RBF)** | **0.7432** | **$38,557.94  | $58,009.39** |

#### Elaboration on Results:

1.  **Baseline Performance (MLR)**: The Multiple Linear Regression model provided a decent baseline but struggled to capture the complex relationships in the California housing market. An R" of ~0.60 suggests that linear combinations of features only account for 60% of the price variance.

2.  **The Benefit of Non-Linearity (Polynomial)**: By introducing second-degree polynomial features (specifically for income and location), we saw a slight improvement in the R" score. This indicates that the relationship between coordinates (Latitude/Longitude) and price is not strictly linear, though Degree 2 was still limited in its flexibility.

3.  **Superiority of SVR**: The Support Vector Regressor with an **RBF Kernel** was the clear winner.
    *   **Why it worked**: Real estate prices in California are highly localized (e.g., high-value clusters in Silicon Valley or coastal Malibu). The RBF kernel is exceptionally good at identifying these 'local' pockets of high value that linear models miss.
    *   **Hyperparameters**: Using a high `C` value (100,000) allowed the model to prioritize accuracy over a smooth boundary, which is necessary when the target values (house prices) have a high range.

#### Conclusion:
The **Support Vector Regression** model is the most suitable for this task, reducing the average prediction error (MAE) by over **$12,000** compared to the standard linear model.

In [ ]:
# Exporting Data for Power BI
# Create a DataFrame with the Actual values and Predictions
results_df = pd.DataFrame({
    'Actual_House_Value': y_test,
    'MLR_Prediction': y_pred_mlr,
    'Poly_Prediction': y_pred_poly,
    'SVR_Prediction': y_pred_svr
})

# Add back the original coordinates and core features for Power BI Visuals
# We use the index of y_test to fetch the original unscaled data from 'df'
original_test_data = df.loc[y_test.index, ['longitude', 'latitude', 'median_income', 'ocean_proximity']]

# Combine them
powerbi_export = pd.concat([original_test_data, results_df], axis=1)

# Calculate the Error (Residuals) for the best model (SVR)
powerbi_export['SVR_Error'] = abs(powerbi_export['Actual_House_Value'] - powerbi_export['SVR_Prediction'])

# Save to CSV
export_filename = "PowerBI_Housing_Results.csv"
powerbi_export.to_csv(export_filename, index=False)

print(f"Data successfully exported to '{export_filename}'!")
print("You can now download this file from the folder icon on the left sidebar.")

Data successfully exported to 'PowerBI_Housing_Results.csv'!
You can now download this file from the folder icon on the left sidebar.
